# Exercice 4 : l'exécution durable

**Mission :** faire en sorte qu'une panne du SOAR **au milieu d'un incident** ne fasse rien perdre, et rien faire deux fois.

Jusqu'ici, l'API exécute le playbook **pendant** la requête HTTP. Trois problèmes :
1. le SIEM attend la fin du playbook pour avoir sa réponse (et si `ip.enrichir` réessaie 3 fois…) ;
2. si le processus plante à l'étape 3, les étapes 1 et 2 sont perdues, et personne ne sait où on en était ;
3. un seul processus fait tout le travail.

Tracecat confie ses exécutions à **Temporal**, un moteur d'exécution durable. On reconstruit l'idée avec **SQLite** :

```
 API ──dépose──► [ file : table executions ] ◄──prend (avec un bail)── travailleur 1, 2, 3…
                                                   │ chaque étape terminée
                                                   ▼
                                     [ table etapes ]  ◄── relue après une panne
```

**Ouvrez `minisoar/file.py`** (environ 100 lignes).

In [1]:
import os, subprocess, sys, time, json
from fastapi.testclient import TestClient
from banc import scenario, environnement as env, rejeu
from minisoar import api, file

os.environ["SOAR_SECRET_WEBHOOK"] = "cle-du-siem-8inf857"
CLE = {"x-soar-cle": "cle-du-siem-8inf857"}
api.LANCER = api.lancer_en_file          # l'API dépose dans la file au lieu d'exécuter
siem = TestClient(api.app)

def travailleur(*options, panne=None):
    """Lance un travailleur dans un AUTRE processus, comme dans un terminal."""
    variables = dict(os.environ, SOAR_PANNE_APRES=panne or "")
    r = subprocess.run([sys.executable, "-m", "minisoar.file", *options], env=variables, capture_output=True, text=True)
    print(r.stdout + r.stderr[-500:])

## 1. Déposer, répondre tout de suite

**Ce qu'on fait :** le SIEM envoie l'alerte de l'intrusion (A06). L'API répond immédiatement « en file » : personne n'a encore rien exécuté.

In [2]:
alertes = scenario.construire()
r = siem.post("/webhooks/siem", json=alertes[5], headers=CLE).json()
print(r)
file.tableau()

{'alerte': 'A06', 'playbook': 'connexion-suspecte', 'execution': '9571dd31', 'statut': 'en file'}


,id,playbook,statut,travailleur
0,9571dd31,connexion-suspecte,en file,None


**Ce qu'on fait :** un travailleur, dans un autre processus, vide la file.

In [3]:
travailleur("--une-fois")
file.etat(r["execution"])["sortie"]

[20:02:16] hp-HP-EliteBook-840-G5-41 prend 9571dd31 (connexion-suspecte)



{'verdict': 'incident',
 'rapport': '203.0.113.77 (interne : False) a fait 21 échec(s) sur admin, oracle, root, webadmin, premier essai à Dec 10 07:05:12. Comptes où elle est entrée : webadmin. Blocage : réussie. Désactivation : réussie.'}

## 2. La panne

**Ce qu'on fait :** on recommence, mais le travailleur **plante juste après avoir bloqué l'IP**, avant d'avoir pu l'écrire dans la base. C'est le pire moment : l'action est faite dans le monde réel, mais le SOAR ne le sait pas.

In [4]:
alertes = scenario.construire()
api.VUES.clear()                          # on oublie les alertes déjà vues (sinon : doublon !)
r = siem.post("/webhooks/siem", json=alertes[5], headers=CLE).json()
travailleur("--une-fois", panne="bloquer")
print("Pare-feu :", env.ips_bloquees())
print("Étapes enregistrées :", [e["ref"] for e in file.etat(r["execution"])["etapes"]])
file.tableau()

[20:02:17] hp-HP-EliteBook-840-G5-48 prend 6f75efc6 (connexion-suspecte)
💥 hp-HP-EliteBook-840-G5-48 plante juste après l'étape « bloquer »

Pare-feu : ['203.0.113.77']
Étapes enregistrées : ['bilan', 'enrichir']


,id,playbook,statut,travailleur
0,6f75efc6,connexion-suspecte,en cours,hp-HP-EliteBook-840-G5-48


👀 **Ce qu'on doit voir :** le travailleur est mort ; l'exécution est toujours « en cours », avec un travailleur qui n'existe plus. L'IP **est** dans le pare-feu, mais l'étape `bloquer` n'est **pas** enregistrée.

**Ce qu'on fait :** attendre que le **bail** expire (5 secondes ici), puis lancer un autre travailleur.

In [5]:
time.sleep(file.BAIL + 1)
travailleur("--une-fois")
print("Étapes :", [(e["ref"], e["statut"], (e["result"] or {}).get("statut")) for e in file.etat(r["execution"])["etapes"]])
[a for a in env.audit() if a["action"] == "bloquer_ip"]

[20:02:24] hp-HP-EliteBook-840-G5-52 prend 6f75efc6 (connexion-suspecte)

Étapes : [('bilan', 'réussie', None), ('enrichir', 'réussie', None), ('bloquer', 'réussie', 'déjà bloquée'), ('desactiver', 'réussie', 'désactivé')]


[{'heure': '08:41:12',
  'acteur': 'soar',
  'action': 'bloquer_ip',
  'cible': '203.0.113.77',
  'resultat': 'bloquée'},
 {'heure': '08:41:12',
  'acteur': 'soar',
  'action': 'bloquer_ip',
  'cible': '203.0.113.77',
  'resultat': 'déjà bloquée'}]

👀 **Ce qu'on doit voir :** le nouveau travailleur **saute** `bilan` et `enrichir` (enregistrées), **refait** `bloquer`, et le pare-feu répond « déjà bloquée ». L'audit montre deux tentatives, mais l'IP n'est qu'une fois dans le pare-feu.

❓ **Question pour la classe :** pourquoi `bloquer` a-t-elle été refaite, alors qu'elle avait réussi ? Et si l'action avait été « envoyer un courriel au client » ?

*Réponse : entre l'action et l'écriture dans la base, il y a toujours un instant où une panne laisse un doute. Le moteur ne peut pas savoir : il refait. C'est une garantie « **au moins une fois** ». Elle n'est sûre que si l'action est **idempotente** (exercice 1). Un courriel serait envoyé deux fois : il faudrait lui donner une **clé d'idempotence** (l'identifiant de l'exécution + la ref), que le service de courriel vérifie.*

❓ **Question pour la classe :** pourquoi un bail, plutôt que « si un travailleur est en cours depuis longtemps, il est mort » ?

*Réponse : un travailleur vivant **renouvelle** son bail à chaque étape (« je suis vivant »). Une étape lente mais saine ne fait donc pas reprendre l'exécution par un autre. Un bail trop court : deux travailleurs font la même chose. Trop long : un incident attend pour rien après une panne.*

## 3. Plusieurs travailleurs

**Ce qu'on fait :** le SIEM envoie les 11 alertes de la matinée d'un coup, et **trois** travailleurs se partagent la file en parallèle.

In [6]:
alertes = scenario.construire()
api.VUES.clear()
for a in alertes:
    source, brut = scenario.a_envoyer(a)
    siem.post(f"/webhooks/{source}", json=brut, headers=CLE)

processus = [subprocess.Popen([sys.executable, "-m", "minisoar.file", "--une-fois"], stdout=subprocess.PIPE, text=True)
             for _ in range(3)]
for p in processus:
    print(p.communicate()[0], end="")
file.tableau()

[20:02:25] hp-HP-EliteBook-840-G5-77 prend 92b4f504 (force-brute)
[20:02:25] hp-HP-EliteBook-840-G5-77 prend e2a0cedb (activite-hote)
[20:02:25] hp-HP-EliteBook-840-G5-78 prend bcb09990 (force-brute)
[20:02:25] hp-HP-EliteBook-840-G5-78 prend 3538980b (connexion-suspecte)
[20:02:25] hp-HP-EliteBook-840-G5-78 prend bcea60d3 (connexion-suspecte)
[20:02:25] hp-HP-EliteBook-840-G5-79 prend 49cd8d5d (connexion-suspecte)
[20:02:25] hp-HP-EliteBook-840-G5-79 prend addab296 (force-brute)
[20:02:25] hp-HP-EliteBook-840-G5-79 prend c1954c47 (activite-hote)
[20:02:25] hp-HP-EliteBook-840-G5-79 prend 9dc375eb (force-brute)
[20:02:25] hp-HP-EliteBook-840-G5-79 prend 6fde30fe (force-brute)
[20:02:25] hp-HP-EliteBook-840-G5-79 prend 67ce3690 (force-brute)


,id,playbook,statut,travailleur
0,92b4f504,force-brute,terminée,hp-HP-EliteBook-840-G5-77
1,bcb09990,force-brute,terminée,hp-HP-EliteBook-840-G5-78
2,49cd8d5d,connexion-suspecte,terminée,hp-HP-EliteBook-840-G5-79
3,3538980b,connexion-suspecte,terminée,hp-HP-EliteBook-840-G5-78
4,addab296,force-brute,terminée,hp-HP-EliteBook-840-G5-79
5,bcea60d3,connexion-suspecte,terminée,hp-HP-EliteBook-840-G5-78
6,c1954c47,activite-hote,terminée,hp-HP-EliteBook-840-G5-79
7,9dc375eb,force-brute,terminée,hp-HP-EliteBook-840-G5-79
8,6fde30fe,force-brute,terminée,hp-HP-EliteBook-840-G5-79
9,67ce3690,force-brute,terminée,hp-HP-EliteBook-840-G5-79


❓ **Question pour la classe :** deux travailleurs auraient-ils pu prendre la même exécution ? Regardez la requête de `prendre`.

*Réponse : non. Un seul `UPDATE … WHERE id = (SELECT …) RETURNING *` choisit ET réserve l'exécution **en une opération atomique** : la base garantit qu'un seul travailleur la gagne. Avec un `SELECT` puis un `UPDATE` séparés, deux travailleurs pourraient lire la même ligne entre les deux.*

## 4. Rejouer la matinée avec la file

**Ce qu'on fait :** le même rejeu qu'à l'exercice 3, mais chaque alerte passe par la file, et un travailleur (ici, dans le notebook) la traite.

In [7]:
api.VUES.clear()

def envoyer(alerte):
    source, brut = scenario.a_envoyer(alerte)
    r = siem.post(f"/webhooks/{source}", json=brut, headers=CLE).json()
    file.travailler(une_fois=True, travailleur="notebook")
    sortie = file.etat(r["execution"])["sortie"] if "execution" in r else {}
    return f"{sortie.get('rapport', r)}\nVERDICT : {sortie.get('verdict', '?')}"

score = rejeu.rejouer(envoyer, "4 - file durable")

07:28:03 A01 R1-force-brute           → attaque sans impact  ✅  (0 s)


07:34:10 A02 R1-force-brute           → attaque sans impact  ✅  (0 s)
08:02:41 A03 R2-succes-apres-echecs   → faux positif         ✅  (0 s)


08:21:07 A04 R3-compte-leurre         → incident             ✅  (0 s)
08:25:15 A05 R1-force-brute           → attaque sans impact  ✅  (0 s)
08:41:12 A06 R2-succes-apres-echecs   → incident             ✅  (0 s)


09:05:00 A07 R4-sortant-suspect       → incident             ✅  (0 s)
09:09:42 A08 R1-force-brute           → attaque sans impact  ✅  (0 s)


09:11:34 A09 R1-force-brute           → attaque sans impact  ✅  (0 s)
09:13:10 A10 R1-force-brute           → attaque sans impact  ✅  (0 s)


09:20:00 A11 R5-cpu                   → faux positif         ❌  (0 s)

Score : 80/100   (verdicts 32/35 · faits 30/35 · réponse 18/30 · pénalités −0)
Faits trouvés : ✅ début de l'attaque à 07:05 · ✅ script x.sh téléchargé · ✅ clé SSH de l'attaquant ajoutée · ✅ crontab de l'attaquant ajoutée · ✅ binaire /tmp/.x/kworkerds · ✅ reconnaissance (uname, whoami, /etc/passwd) · ❌ source de l'injection (198.51.100.23) signalée
Actions : ✅ IP de l'attaquant bloquée · ✅ compte webadmin désactivé · ❌ clé SSH de l'attaquant retirée · ❌ crontab de l'attaquant retirée · ✅ mineur arrêté
Rapport complet : resultats/rapport - 4 - file durable.md   (0.0 min)


❓ **Question pour la classe :** le score est le même qu'à l'exercice 3. Alors à quoi a servi cet exercice ?

*Réponse : la durabilité n'améliore pas la **qualité** de la réponse ; elle garantit qu'elle **ira au bout**, même si le SOAR plante, et qu'elle peut **monter en charge** (plus de travailleurs). Un score ne mesure pas tout : celui-ci ne simule aucune panne.*

## 5. À vous : le bouton d'arrêt

Un playbook mal écrit tourne en boucle, ou bloque des IP légitimes : l'analyste doit pouvoir **annuler** une exécution. Dans `minisoar/file.py`, écrivez `annuler(id_)` : l'exécution passe au statut `annulée` si elle est encore `en file` ; elle renvoie `True` si elle a été annulée, `False` sinon. Les travailleurs ne doivent plus la prendre.

*Pour aller plus loin :* ajoutez la route `POST /executions/{id_}/annuler` dans `api.py`.

In [ ]:
from minisoar import file
scenario.construire()
id_ = file.deposer("force-brute", alertes[0])
assert file.annuler(id_) is True
assert file.prendre("test") is None, "un travailleur a pris une exécution annulée !"
assert file.etat(id_)["statut"] == "annulée"
assert file.annuler(id_) is False
print("✅ Le bouton d'arrêt fonctionne.")

*Réponse :*
```python
def annuler(id_: str) -> bool:
    """Annule une exécution qui n'a pas encore commencé."""
    n = base().execute("UPDATE executions SET statut = 'annulée' WHERE id = ? AND statut = 'en file'", (id_,)).rowcount
    return n == 1
```
*Et une exécution déjà « en cours » ? Il faudrait que le moteur vérifie le statut **entre deux étapes** et s'arrête proprement : c'est ce que fait Temporal avec ses signaux d'annulation.*

## Ce qu'il faut retenir
- L'API **dépose** et répond tout de suite ; des **travailleurs** exécutent. On peut en ajouter pour monter en charge.
- Chaque étape terminée est **enregistrée** : après une panne, on **reprend**, sans refaire ce qui est fait.
- Entre l'action et l'enregistrement, une panne laisse un doute : la garantie est « **au moins une fois** », donc les actions doivent être **idempotentes**.
- Réserver une tâche doit être **atomique**, et protégé par un **bail** que le travailleur renouvelle.
- Prochain exercice : les **cas**, et un humain dans la boucle.